# Orders Agent & Chatbot

A **custom agentic chatbot** that can **query** and **update** laptop orders using LangGraph.

**Architecture diagram & design doc:** [README_ORDERS_AGENT.md](../README_ORDERS_AGENT.md)

---

## What This Example Demonstrates

| Component | Purpose |
|-----------|---------|
| **Custom StateGraph** | Manually wired ReAct loop (LLM node → conditional edge → tool node → loop back) |
| **OrdersAgent class** | Encapsulates graph, LLM, tools, and memory in one reusable class |
| **get_order_details tool** | READ: Returns order details for an order ID (exact match on DataFrame) |
| **update_quantity tool** | WRITE: Updates laptop quantity in an order |
| **MemorySaver** | Conversation memory per `thread_id` — "add one more" remembers which order |

---

## How It Works

1. **User asks** (e.g. "Show me order ORD-7311")
2. **Orders LLM** analyzes the prompt, determines the tool to call and its parameters
3. **Conditional edge** checks: is the next action a tool call?
4. **Order Tools** executes the tool, writes results back to agent state
5. **Loop back** to Orders LLM — it reviews results. If sufficient info → final answer. If not → another tool call.
6. **END** when the final answer is ready, returned to the chatbot

All data flows through the **agent state** (messages list), never through edges.

---

## Graph

```
START → orders_llm → [is_tool_call?] → orders_tools (yes) → orders_llm
                                       → END (no — final answer)
```

## 1. Setup Models

Load the LLM using `ChatOpenAI` with the API key from `.env`.

**Key concept:** `load_dotenv()` reads your `.env` file and sets `OPENAI_API_KEY` as an environment variable. The `ChatOpenAI` constructor picks it up automatically — no need to pass the key explicitly.

> To use Azure OpenAI, replace `ChatOpenAI` with `AzureChatOpenAI` and set `AZURE_OPENAI_API_KEY` + `AZURE_OPENAI_ENDPOINT` in `.env`.

In [ ]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

# Load API key from .env file
load_dotenv()

# Setup the LLM
# temperature=0 → deterministic output (same input = same output)
model = ChatOpenAI(model="gpt-4o-mini", temperature=0)

## 2. Setup Tools for Custom Agent

We define **two function tools** that the agent can call:

| Tool | Type | What It Does |
|------|------|--------------|
| `get_order_details` | **Query (read)** | Returns product name, quantity, and delivery date for an order ID |
| `update_quantity` | **Action (write)** | Updates the quantity of laptops in an order |

The `@tool` decorator turns each Python function into a LangChain tool. The LLM reads the **docstring** to decide when to call it and what arguments to provide.

### Data Source

The orders "database" is a **Pandas DataFrame** loaded from `data/laptop_orders.csv`. In a real application this would be an RDBMS (PostgreSQL, MySQL, etc.).

In [ ]:
import pandas as pd

# Load the laptop orders CSV into a Pandas DataFrame
product_orders_df = pd.read_csv("../data/laptop_orders.csv")
print(product_orders_df)

In [ ]:
from langchain_core.tools import tool


@tool
def get_order_details(order_id: str) -> str:
    """
    Return details about a laptop order given an order ID.
    Performs an exact match between the input order ID and available order IDs.
    If found: returns product ordered, quantity ordered, and delivery date.
    If NOT found: returns -1.
    """
    match_order_df = product_orders_df[
        product_orders_df["Order ID"] == order_id
    ]

    if len(match_order_df) == 0:
        return -1
    else:
        return match_order_df.iloc[0].to_dict()


@tool
def update_quantity(order_id: str, new_quantity: int) -> bool:
    """
    Update the quantity of products (laptops) ordered for a given order ID.
    If the order exists: updates the quantity and returns True.
    If there is NO matching order: returns False.
    """
    global product_orders_df

    match_order_df = product_orders_df[
        product_orders_df["Order ID"] == order_id
    ]

    if len(match_order_df) == 0:
        return False
    else:
        product_orders_df.loc[
            product_orders_df["Order ID"] == order_id,
            "Quantity Ordered"
        ] = new_quantity
        return True

## 3. Setup the Custom Orders Agent

This is where we build the **LangGraph StateGraph manually** — the same ReAct pattern as `create_react_agent`, but with full control over every node and edge.

### Key Components

| Component | Role |
|-----------|------|
| `OrdersAgentState` | Agent state — holds the `messages` list. `operator.add` means new messages are **appended**, not replaced. |
| `OrdersAgent.__init__` | Builds the graph: adds nodes, conditional edges, entry point, and compiles with MemorySaver. |
| `call_llm` | **LLM node** — prepends system prompt, invokes the model, returns the response. |
| `is_tool_call` | **Conditional edge function** — returns `True` if the LLM wants to call a tool, `False` if the answer is ready. |
| `call_tools` | **Tool node** — executes the requested tools, wraps results in `ToolMessage`, returns to state. |

### Graph Wiring

```
entry_point → orders_llm
orders_llm  → [is_tool_call?] → True:  orders_tools
                                → False: END
orders_tools → orders_llm  (loop back)
```

### Why Build Manually?

`create_react_agent` is convenient but opaque. Building manually lets you:
- Add custom nodes (e.g., logging, validation, human-in-the-loop)
- Control the conditional routing logic
- Add a `debug` flag for step-by-step output
- Use a custom agent state with extra fields beyond just `messages`

In [ ]:
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver
from typing import TypedDict, Annotated
import operator
from langchain_core.messages import AnyMessage, SystemMessage, HumanMessage, ToolMessage


# Agent State: holds the conversation messages.
# operator.add means new messages are APPENDED to the list (not replaced).
class OrdersAgentState(TypedDict):
    messages: Annotated[list[AnyMessage], operator.add]


class OrdersAgent:
    """Custom agent that manages laptop order queries and updates."""

    def __init__(self, model, tools, system_prompt, debug=False):
        self.system_prompt = system_prompt
        self.debug = debug

        # Build the graph
        agent_graph = StateGraph(OrdersAgentState)
        agent_graph.add_node("orders_llm", self.call_llm)
        agent_graph.add_node("orders_tools", self.call_tools)
        agent_graph.add_conditional_edges(
            "orders_llm",
            self.is_tool_call,
            {True: "orders_tools", False: END}
        )
        agent_graph.add_edge("orders_tools", "orders_llm")
        agent_graph.set_entry_point("orders_llm")

        # Add conversation memory
        self.memory = MemorySaver()
        self.agent_graph = agent_graph.compile(checkpointer=self.memory)

        # Store tools for lookup
        self.tools = {t.name: t for t in tools}
        if self.debug:
            print("\nTools loaded:", list(self.tools.keys()))

        # Bind tools to the model
        self.model = model.bind_tools(tools)

    def call_llm(self, state: OrdersAgentState):
        """LLM node: Analyze prompt, determine actions, review results."""
        messages = state["messages"]
        if self.system_prompt:
            messages = [SystemMessage(content=self.system_prompt)] + messages
        result = self.model.invoke(messages)
        if self.debug:
            print(f"\nLLM returned: {result}")
        return {"messages": [result]}

    def is_tool_call(self, state: OrdersAgentState):
        """Conditional edge: True if LLM wants to call a tool, False if done."""
        last_message = state["messages"][-1]
        return len(last_message.tool_calls) > 0

    def call_tools(self, state: OrdersAgentState):
        """Tool node: Execute requested tools and return results."""
        tool_calls = state["messages"][-1].tool_calls
        results = []

        for tc in tool_calls:
            if tc["name"] not in self.tools:
                print(f"Unknown tool: {tc['name']}")
                result = "Invalid tool found. Please retry."
            else:
                result = self.tools[tc["name"]].invoke(tc["args"])

            results.append(
                ToolMessage(
                    tool_call_id=tc["id"],
                    name=tc["name"],
                    content=str(result),
                )
            )

        if self.debug:
            print(f"\nTools returned: {results}")
        return {"messages": results}

### Create the Agent Instance

The **system prompt** defines the chatbot's persona:
- Manages laptop orders
- Does NOT reveal info about other orders (privacy)
- Handles small talk professionally

In [ ]:
system_prompt = """
You are a professional chatbot that manages orders for laptops sold by our company.
The tools allow for retrieving order details as well as updating order quantity.
Do NOT reveal information about other orders than the one requested.
You will handle small talk and greetings by producing professional responses.
"""

orders_agent = OrdersAgent(
    model,
    [get_order_details, update_quantity],
    system_prompt,
    debug=False,
)

### Visualize the Agent Graph

LangGraph can render the compiled graph as an image. This shows the nodes, edges, and conditional routing.

In [ ]:
from IPython.display import Image

Image(orders_agent.agent_graph.get_graph().draw_mermaid_png())

## 4. Setup and Execute the Orders Chatbot

We simulate a **multi-turn conversation** between a user and the chatbot. All messages share the same `thread_id`, so the agent **remembers context** across turns.

### Conversation Flow

| Turn | User Says | What Happens |
|------|-----------|--------------|
| 1 | "How are you doing?" | Small talk — no tools needed, agent responds directly |
| 2 | "Show me order ORD-7311" | Agent calls `get_order_details("ORD-7311")` → returns NanoEdge Flex, qty 2 |
| 3 | "Add one more of that laptop" | Agent remembers ORD-7311, calls `update_quantity("ORD-7311", 3)` |
| 4 | "Show me the details again" | Agent calls `get_order_details("ORD-7311")` → now shows qty 3 |
| 5 | "What about ORD-9999?" | Agent calls `get_order_details("ORD-9999")` → returns -1 (not found) |
| 6 | "Bye" | Small talk — professional farewell |

### Key Observations

- **Turn 3** works because of conversation memory: the agent knows "that laptop" = ORD-7311
- **Turn 4** confirms the update actually modified the DataFrame
- **Turn 5** shows graceful handling of missing orders

In [ ]:
import uuid

# Simulated user messages
user_inputs = [
    "How are you doing?",
    "Please show me the details of the order ORD-7311",
    "Can you add one more of that laptop to the order?",
    "Can you show me the details again?",
    "What about order ORD-9999?",
    "Bye",
]

# Create a unique thread for this conversation
config = {"configurable": {"thread_id": str(uuid.uuid4())}}

for user_input in user_inputs:
    print(f"{'─' * 40}\nUSER  : {user_input}")
    user_message = {"messages": [HumanMessage(user_input)]}
    ai_response = orders_agent.agent_graph.invoke(user_message, config=config)
    print(f"AGENT : {ai_response['messages'][-1].content}\n")

### Verify the DataFrame Was Updated

The `update_quantity` tool modifies the DataFrame in-place. Let's confirm ORD-7311 now has quantity 3:

In [ ]:
print(product_orders_df)

## Summary

| Concept | How It Appears in This Example |
|---------|-------------------------------|
| **Custom StateGraph** | Manually built graph with `add_node`, `add_conditional_edges`, `set_entry_point` |
| **LLM node** | `call_llm` — sends messages to the model, gets actions or final answer |
| **Tool node** | `call_tools` — executes `get_order_details` / `update_quantity` |
| **Conditional edge** | `is_tool_call` — routes to tools or END based on `tool_calls` in last message |
| **Agent State** | `OrdersAgentState` — messages list with `operator.add` for append semantics |
| **MemorySaver** | Conversation memory per `thread_id` — "add one more" remembers which order |
| **System prompt** | Defines persona: manages orders, doesn't leak info, handles small talk |
| **@tool decorator** | Turns Python functions into LLM-callable tools with auto-generated schemas |
| **bind_tools** | Attaches tool schemas to the LLM so it can request tool calls |

### Manual Build vs. `create_react_agent`

| | Manual (this example) | `create_react_agent` (Example 8) |
|---|---|---|
| **Control** | Full control over nodes, edges, routing | Opaque — one-liner setup |
| **Customization** | Easy to add logging, validation, human-in-the-loop nodes | Limited to parameters |
| **Debug** | Custom `debug` flag in the class | Built-in `debug=True` |
| **Code** | More code, but more transparent | Less code, faster to set up |
| **When to use** | Production agents, complex routing | Prototyping, simple agents |